In [3]:
import pandas as pd
import numpy as np

# Load patient-stratified split metadata
df = pd.read_csv("data/metadata_splits/metadata_splits.csv")

print("==================================================")
print("1. OVERALL DATASET & CLASS IMBALANCE AUDIT")
print("==================================================")
total_samples = len(df)
total_patients = df['patient_id'].nunique()
malignant_count = df['target'].sum()
benign_count = total_samples - malignant_count
pos_ratio = (malignant_count / total_samples) * 100

print(f"Total Samples           : {total_samples:,}")
print(f"Total Unique Patients   : {total_patients:,}")
print(f"Benign Cases (0)        : {benign_count:,} ({100 - pos_ratio:.2f}%)")
print(f"Malignant Cases (1)     : {malignant_count:,} ({pos_ratio:.2f}%)")
print(f"Imbalance Ratio         : 1:{benign_count / max(1, malignant_count):.1f}")

print("\n==================================================")
print("2. TABULAR FEATURE MISSINGNESS ANALYSIS")
print("==================================================")
missing = df.isnull().sum()
missing_pct = (missing / total_samples) * 100
missing_df = pd.DataFrame({'Missing_Count': missing, 'Missing_Percent': missing_pct})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values(by='Missing_Percent', ascending=False)

if missing_df.empty:
    print("Zero missing values across metadata features.")
else:
    print(missing_df.to_string())

print("\n==================================================")
print("3. DEMOGRAPHIC & CLINICAL BIAS AUDIT")
print("==================================================")
if 'sex' in df.columns:
    print("\n-- Target Distribution by Sex --")
    print(pd.crosstab(df['sex'].fillna('Missing'), df['target'], normalize='index') * 100)

if 'anatom_site_general' in df.columns:
    print("\n-- Target Distribution by Anatomical Site --")
    print(pd.crosstab(df['anatom_site_general'].fillna('Missing'), df['target'], normalize='index') * 100)

if 'age_approx' in df.columns:
    print("\n-- Age Summary statistics by Malignancy --")
    print(df.groupby('target')['age_approx'].describe())

print("\n==================================================")
print("4. PATIENT LESION DENSITY DISTRIBUTION")
print("==================================================")
lesions_per_patient = df.groupby('patient_id')['isic_id'].count()
print(f"Mean Lesions per Patient : {lesions_per_patient.mean():.2f}")
print(f"Max Lesions on 1 Patient : {lesions_per_patient.max()}")
print(f"Single-Lesion Patients   : {(lesions_per_patient == 1).sum()} ({(lesions_per_patient == 1).mean()*100:.1f}%)")

/tmp/ipykernel_269464/3849713061.py:5: DtypeWarning: Columns (0: iddx_5, 1: mel_mitotic_index) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/metadata_splits/metadata_splits.csv")


1. OVERALL DATASET & CLASS IMBALANCE AUDIT
Total Samples           : 401,059
Total Unique Patients   : 1,042
Benign Cases (0)        : 400,666 (99.90%)
Malignant Cases (1)     : 393 (0.10%)
Imbalance Ratio         : 1:1019.5

2. TABULAR FEATURE MISSINGNESS ANALYSIS
                     Missing_Count  Missing_Percent
iddx_5                      401058        99.999751
mel_mitotic_index           401006        99.986785
mel_thick_mm                400996        99.984292
iddx_4                      400508        99.862614
iddx_3                      399994        99.734453
iddx_2                      399991        99.733705
lesion_id                   379001        94.500061
sex                          11517         2.871647
anatom_site_general           5756         1.435200
age_approx                    2798         0.697653

3. DEMOGRAPHIC & CLINICAL BIAS AUDIT

-- Target Distribution by Sex --
target           0         1
sex                         
Missing  99.913172  0.086828
fem